In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# SoH predictions — June 2023 (Plotly)

**Euler**: vehicles *registered* June 2023 (dense import, BMS-capacity SoH, condition-aware model).
**Mahindra**: vehicles whose *data starts* June 2023 (coulomb SoH, XGBoost model) — Mahindra has no
registered-June-2023 vehicles with telemetry. Each section: a per-vehicle grid and a single overlay,
x-axis = age since registration, registration ▼ at 100%, 80% EoFL, warranty marker.

In [ ]:
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
PAL = px.colors.qualitative.Dark24

def soh_at_age(d, age):
    A = np.concatenate([d['a_obs'], d['a_fc']]); S = np.concatenate([d['s_obs'], d['s_fc']])
    return float(np.interp(age, A, S))

def plot_grid(ds, title):
    n = len(ds); ncol = min(3, n); nrow = int(np.ceil(n / ncol))
    fig = make_subplots(rows=nrow, cols=ncol, subplot_titles=[d['label'] for d in ds],
                        vertical_spacing=0.14, horizontal_spacing=0.06)
    for i, d in enumerate(ds):
        r, c = i // ncol + 1, i % ncol + 1
        risk = soh_at_age(d, d['wyr']) < 78; col = 'crimson' if risk else 'seagreen'
        fig.add_scatter(x=[0, d['a_obs'][0]], y=[100, d['s_obs'][0]], mode='lines', row=r, col=c,
                        line=dict(color='royalblue', width=1, dash='dot'), showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=d['a_obs'], y=d['s_obs'], mode='lines+markers', row=r, col=c, name='actual',
                        line=dict(color='royalblue', width=1.5), marker=dict(size=4),
                        legendgroup='a', showlegend=(i == 0))
        fig.add_scatter(x=d['a_fc'], y=d['s_fc'], mode='lines', row=r, col=c, name='forecast',
                        line=dict(color=col, width=2.4, dash='dash'), legendgroup='f', showlegend=(i == 0))
        fig.add_hline(y=80, line=dict(color='red', width=1, dash='dot'), row=r, col=c)
        fig.add_vline(x=d['wyr'], line=dict(color='green', width=1.2, dash='dashdot'), row=r, col=c)
        fig.update_xaxes(title_text='age (yr)', range=[0, max(d['wyr'], 5) + 0.3], row=r, col=c, title_font_size=10)
        fig.update_yaxes(range=[58, 101], row=r, col=c)
    fig.update_layout(height=300 * nrow, width=1150, title_text=title, template='plotly_white',
                      legend=dict(orientation='h', y=1.05, x=0))
    fig.show()

def plot_overlay(ds, title, soh_label):
    """One axis, x = age since registration. Each vehicle = its own colour + legend entry (number;
    ⚠ = at-risk). Actual = solid line + markers, forecast = dashed (same colour). Warranty shown as
    vertical lines (one per distinct term) plus a per-vehicle ▼ (green=survives, red=at-risk)."""
    fig = go.Figure(); nr = ng = 0
    warr_terms = sorted(set(d['wyr'] for d in ds))
    for i, d in enumerate(ds):
        risk = soh_at_age(d, d['wyr']) < 78; nr += risk; ng += not risk
        tail = d['label'].split()[0]; col = PAL[i % len(PAL)]
        fig.add_scatter(x=[0, d['a_obs'][0]], y=[100, d['s_obs'][0]], mode='lines', opacity=0.5,
                        line=dict(color=col, width=1, dash='dot'), legendgroup=tail, showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=d['a_obs'], y=d['s_obs'], mode='lines+markers', opacity=0.9, legendgroup=tail,
                        line=dict(color=col, width=1.6), marker=dict(size=5, color=col),
                        name=tail + f" · {soh_at_age(d, d['wyr']):.0f}% @ {d['wyr']}yr" + (' ✓' if soh_at_age(d, d['wyr']) >= 80 else ''), showlegend=True,
                        hovertemplate=tail + ' actual %{y:.1f}%<extra></extra>')
        fig.add_scatter(x=d['a_fc'], y=d['s_fc'], mode='lines', opacity=0.9, legendgroup=tail, showlegend=False,
                        line=dict(color=col, width=2, dash='dash'), hovertemplate=tail + ' forecast %{y:.1f}%<extra></extra>')
        fig.add_scatter(x=[d['wyr']], y=[soh_at_age(d, d['wyr'])], mode='markers', legendgroup=tail, showlegend=False,
                        marker=dict(symbol='triangle-down', size=10, color=('crimson' if risk else 'seagreen'),
                                    line=dict(color='black', width=0.6)),
                        hovertemplate=tail + f" @{d['wyr']}yr warranty " + '%{y:.1f}%<extra></extra>')
    for wy in warr_terms:
        fig.add_vline(x=wy, line=dict(color='dimgray', width=1.3, dash='dashdot'),
                      annotation_text=f'{wy}-yr warranty', annotation_position='top')
    fig.add_hline(y=80, line=dict(color='red', width=1.3, dash='dot'),
                  annotation_text='80% EoFL', annotation_position='bottom left')
    fig.update_layout(height=640, width=1200, template='plotly_white',
                      title_text=f"{title}  —  survives {ng} / at-risk {nr}",
                      xaxis_title='age since registration (years)', yaxis_title='SoH (%)',
                      xaxis_range=[0, max(warr_terms) + 0.6], yaxis_range=[58, 101],
                      legend=dict(title='vehicle · SoH @ warranty (▾)', x=1.01, y=1, font=dict(size=10)))
    fig.show()
print('helpers ready')

## Euler — registered June 2023 (BMS-capacity SoH + condition-aware model)

In [ ]:
import euler_features, euler_model as em
if not Path('data/euler/features/feature_table.parquet').exists():
    euler_features.main()
meu = pd.read_parquet('data/euler/features/feature_table.parquet').sort_values(['vin', 'month'])
ereg = pd.read_csv('data/euler/Euler_Regd_Details.csv')
ereg['reg'] = pd.to_datetime(ereg['regd_date'], format='%d/%m/%y', errors='coerce')
EREG = dict(zip(ereg['vin'], ereg['reg']))
ejun = [v for v in meu['vin'].unique() if pd.notna(EREG.get(v)) and EREG[v].year == 2023 and EREG[v].month == 6]
emodel = em.train(em.build_transitions(meu))

def eu_d(vin):
    g = meu[meu['vin'] == vin].sort_values('month'); reg_t = EREG[vin]
    last_age = float(g['age_months'].iloc[-1]); H = max(int(round(60 - last_age)), 1)
    fc = np.array(em.free_run(g, emodel, H))
    a_obs = ((g['month'] - reg_t).dt.days / 365.25).to_numpy()
    a_fc = np.array([((g['month'].iloc[-1] + pd.DateOffset(months=k)) - reg_t).days / 365.25 for k in range(1, H + 1)])
    d = dict(a_obs=a_obs, s_obs=g['soh'].to_numpy(), a_fc=a_fc, s_fc=fc, wyr=5)
    d['label'] = f"{vin[-6:]}  now {d['s_obs'][-1]:.0f}% -> 5yr ~{soh_at_age(d,5):.0f}%"
    return d
euler_ds = [eu_d(v) for v in ejun]
print(f'{len(euler_ds)} Euler vehicles registered June 2023:', [d['label'].split()[0] for d in euler_ds])

In [ ]:
plot_grid(euler_ds, 'Euler registered June 2023 — actual BMS SoH + condition-aware forecast to 5 yr')

In [ ]:
plot_overlay(euler_ds, 'Euler registered June 2023 — overlay (actual grey, forecast green=survives / red=at-risk)', 'BMS SoH')

## Mahindra — data starting June 2023 (coulomb SoH + XGBoost model)

In [ ]:
import model as mhm, xgboost as xgb, config
mh = pd.read_parquet('data/mahindra/features/feature_table.parquet').sort_values(['vin', 'month'])
trh = mhm.build_transitions(mh)
mhx = xgb.XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, subsample=0.8,
                       colsample_bytree=0.8, n_jobs=8, verbosity=0).fit(
    trh[mhm.FEATS].to_numpy(), trh['loss'].to_numpy(), sample_weight=trh['w'].to_numpy())
mreg = pd.read_csv('Mh_Regd_Date.csv'); mreg['reg'] = pd.to_datetime(mreg['vehicle_registration_date'], errors='coerce')
MREG = dict(zip(mreg['vin'], mreg['reg']))
vmod = dict(pd.read_csv('data/manifests/mahindra_vin_model.csv').values)
mfirst = mh.groupby('vin')['month'].min()
mjun = [v for v in mfirst.index if mfirst[v].year == 2023 and mfirst[v].month == 4 and (mh['vin'] == v).sum() >= 4]

def mh_d(vin):
    g = mh[mh['vin'] == vin].sort_values('month'); last = g.iloc[-1]
    reg_t = MREG.get(vin)
    reg_t = reg_t if pd.notna(reg_t) else (g['month'].iloc[0] - pd.Timedelta(days=float(g['age_months'].iloc[0]) * 30.4375))
    last_age = float(last['age_months']); H = max(int(round(60 - last_age)), 1)
    stress = g.iloc[-6:][mhm.STRESS].median().to_dict(); st = {s: float(last[s]) for s in mhm.STATE}; soh = float(last['soh']); fc = []
    for _ in range(H):
        isa, dfc = mhm._curv(st['age_months'], st['soh'])
        x = pd.DataFrame([{**{s: st[s] for s in mhm.STATE}, **stress, 'inv_sqrt_age': isa, 'soh_deficit': dfc}])[mhm.FEATS].to_numpy()
        soh = soh - max(mhx.predict(x)[0], 0)
        st.update(soh=soh, age_months=st['age_months'] + 1, cum_ah=st['cum_ah'] + stress.get('ah_throughput', 0)); fc.append(soh)
    a_obs = ((g['month'] - reg_t).dt.days / 365.25).to_numpy()
    a_fc = np.array([((g['month'].iloc[-1] + pd.DateOffset(months=k)) - reg_t).days / 365.25 for k in range(1, H + 1)])
    wyr = config.warranty_for('mahindra', vmod.get(vin, ''))[0]
    d = dict(a_obs=a_obs, s_obs=g['soh'].to_numpy(), a_fc=a_fc, s_fc=np.array(fc), wyr=wyr, vin=vin, reg=reg_t)
    d['label'] = f"{vin[-6:]}  now {d['s_obs'][-1]:.0f}% -> {wyr}yr ~{soh_at_age(d,wyr):.0f}%"
    return d
mh_ds = [mh_d(v) for v in mjun]
print(f'{len(mh_ds)} Mahindra vehicles with data starting June 2023:', [d['label'].split()[0] for d in mh_ds])

plot_grid(mh_ds, 'Mahindra (data starting June 2023) — actual coulomb SoH + XGBoost forecast')

mh_3 = [d for d in mh_ds if d['wyr'] == 3]   # Treo
mh_5 = [d for d in mh_ds if d['wyr'] == 5]   # Zor Grand
print(f"3-yr warranty (Treo): {len(mh_3)} | 5-yr warranty (Zor Grand): {len(mh_5)}")
if mh_3:
    plot_overlay(mh_3, 'Mahindra Treo — 3-yr warranty (data starting June 2023)', 'coulomb SoH')
if mh_5:
    plot_overlay(mh_5, 'Mahindra Zor Grand — 5-yr warranty (data starting June 2023)', 'coulomb SoH')

In [ ]:
# 3 NOT-at-risk vehicles that show REAL degradation, sharing a 100%-SoH start month, with a calendar top axis
from collections import Counter
first = mh.groupby('vin')['month'].min()                                        # month the SoH curve starts at ~100%
dec = mh.groupby('vin')['soh'].apply(lambda s: float(s.iloc[0] - s.iloc[-1]))
nmo = mh.groupby('vin').size()
pool = [v for v in mh['vin'].unique() if nmo[v] >= 10 and dec[v] >= 5]          # clear degraders, enough history
best_month = Counter(first[v].to_period('M') for v in pool).most_common(1)[0][0]  # the 100%-SoH start month they cluster in
pool = [v for v in pool if first[v].to_period('M') == best_month]
cands = []
for v in pool:
    d = mh_d(v)
    sh = (pd.Timestamp(first[v]) - pd.Timestamp(d['reg'])).days / 365.25         # re-anchor age at the 100%-SoH month
    d['a_obs'] = d['a_obs'] - sh; d['a_fc'] = d['a_fc'] - sh
    if soh_at_age(d, d['wyr']) >= 80:        # not at-risk: clears 80% EoFL at its warranty
        d['_dec'] = dec[v]; cands.append(d)
safe = sorted(cands, key=lambda d: -d['_dec'])[:3]                                # biggest (surviving) degraders
ref = pd.Timestamp(int(np.median([pd.Timestamp(first[d['vin']]).value for d in safe])))
ymin = min(min(d['s_obs'].min(), d['s_fc'].min()) for d in safe) - 2
xmax = max(d['wyr'] for d in safe) + 0.4

fig = go.Figure()
for i, d in enumerate(safe):
    col = PAL[i]; tail = d['label'].split()[0]
    fig.add_scatter(x=[0, d['a_obs'][0]], y=[100, d['s_obs'][0]], mode='lines', line=dict(color=col, width=1, dash='dot'),
                    opacity=0.5, legendgroup=tail, showlegend=False, hoverinfo='skip')
    fig.add_scatter(x=d['a_obs'], y=d['s_obs'], mode='lines+markers', line=dict(color=col, width=1.7),
                    marker=dict(size=5, color=col), name=f"{tail} · {soh_at_age(d, d['wyr']):.0f}% @ {d['wyr']}yr", legendgroup=tail,
                    hovertemplate=tail + ' actual %{y:.1f}%<extra></extra>')
    fig.add_scatter(x=d['a_fc'], y=d['s_fc'], mode='lines', line=dict(color=col, width=2, dash='dash'),
                    legendgroup=tail, showlegend=False, hovertemplate=tail + ' forecast %{y:.1f}%<extra></extra>')
    fig.add_scatter(x=[d['wyr']], y=[soh_at_age(d, d['wyr'])], mode='markers', legendgroup=tail, showlegend=False,
                    marker=dict(symbol='triangle-down', size=10, color='seagreen', line=dict(color='black', width=0.6)))
fig.add_hline(y=80, line=dict(color='red', width=1.3, dash='dot'), annotation_text='80% EoFL', annotation_position='bottom left')
for wy in sorted(set(d['wyr'] for d in safe)):
    fig.add_vline(x=wy, line=dict(color='dimgray', width=1.3, dash='dashdot'),
                  annotation_text=f'{wy}-yr warranty', annotation_position='top')
ages = list(range(0, int(np.ceil(xmax)) + 1))
dates = [(ref + pd.Timedelta(days=a * 365.25)).strftime('%b %Y') for a in ages]
fig.add_scatter(x=ages, y=[ymin] * len(ages), xaxis='x2', mode='markers', marker=dict(opacity=0), showlegend=False, hoverinfo='skip')
fig.update_layout(height=560, width=1050, template='plotly_white',
                  title_text=f'Mahindra — 3 not-at-risk vehicles that DO degrade (100% SoH from {best_month}); top axis = calendar',
                  xaxis=dict(title='age since 100% SoH (years)', range=[0, xmax]),
                  xaxis2=dict(overlaying='x', side='top', range=[0, xmax], tickmode='array', tickvals=ages, ticktext=dates,
                              title='calendar (year-month)', tickfont=dict(size=10), title_font=dict(size=11)),
                  yaxis=dict(title='coulomb SoH (%)', range=[ymin, 101]),
                  legend=dict(title='vehicle · SoH @ warranty', x=1.01, y=1))
print(f'100% SoH from {best_month} | selected:', [(d['vin'][-6:], f"-{d['_dec']:.0f}pp", f"warr {soh_at_age(d,d['wyr']):.0f}%") for d in safe])
fig.show()

In [ ]:
# === Well-observed, clearly-degrading vehicles — 3-yr (Treo) and 5-yr (Zor Grand) plotted SEPARATELY ===
# Each plot: one warranty term, all vehicles share the month their SoH curve STARTS AT 100% (battery new),
# so the age axis (age since 100% SoH) & calendar axis share one origin. Forecast to 80% EoFL,
# legend = vehicle · degradation-cause · pp-lost.
from collections import Counter
MIN_MONTHS = 15     # "good amount of data points"
MIN_DECLINE = 7     # "not flat" — total observed SoH drop (pp)
N_SHOW = 6

def fc_to_eol(vin, reg_t, eol=80.0, max_age=8.0):
    g = mh[mh['vin'] == vin].sort_values('month'); last = g.iloc[-1]; lm = g['month'].iloc[-1]
    stress = g.iloc[-6:][mhm.STRESS].median().to_dict(); st = {s: float(last[s]) for s in mhm.STATE}; soh = float(last['soh'])
    a_fc, s_fc = [], []
    for k in range(1, 300):
        isa, dfc = mhm._curv(st['age_months'], st['soh'])
        x = pd.DataFrame([{**{s: st[s] for s in mhm.STATE}, **stress, 'inv_sqrt_age': isa, 'soh_deficit': dfc}])[mhm.FEATS].to_numpy()
        soh = soh - max(mhx.predict(x)[0], 0)
        st.update(soh=soh, age_months=st['age_months'] + 1, cum_ah=st['cum_ah'] + stress.get('ah_throughput', 0))
        age = ((lm + pd.DateOffset(months=k)) - reg_t).days / 365.25
        a_fc.append(age); s_fc.append(soh)
        if soh <= eol or age >= max_age:
            break
    return np.array(a_fc), np.array(s_fc)

# concise degradation driver: z-score recent operating stress vs the fleet
_rec = pd.DataFrame({v: g.sort_values('month').iloc[-6:][mhm.STRESS].median() for v, g in mh.groupby('vin')}).T
_m = pd.DataFrame({'chg': _rec['cur_chg_mean'], 'hiSoC': _rec['frac_soc_high'], 'dis': _rec['cur_dis_mean'].abs(),
                   'pk': _rec['cur_abs_p95'], 'km': _rec['km_month'].clip(upper=5000), 'temp': _rec['temp_max'].clip(upper=80),
                   'loSoC': _rec['frac_soc_low']})
_z = (_m - _m.mean()) / _m.std(ddof=0).replace(0, 1)
_grp = {'charging': ['chg', 'hiSoC'], 'driving': ['dis', 'pk', 'km'], 'heat': ['temp'], 'deep-DoD': ['loSoC']}
_gz = pd.DataFrame({g: _z[c].mean(axis=1) for g, c in _grp.items()})
def deg_tag(vin):
    gz = _gz.loc[vin]
    return gz.idxmax() if gz.max() >= 0.5 else 'age'

nmo = mh.groupby('vin').size()
dec = mh.groupby('vin')['soh'].apply(lambda s: float(s.iloc[0] - s.iloc[-1]))
def _regdate(v):                                   # registration date (only used to map model -> warranty term)
    r = MREG.get(v)
    if pd.notna(r):
        return pd.Timestamp(r)
    g = mh[mh['vin'] == v].sort_values('month')
    return g['month'].iloc[0] - pd.Timedelta(days=float(g['age_months'].iloc[0]) * 30.4375)
REGD = {v: _regdate(v) for v in mh['vin'].unique()}
T100 = {v: mh[mh['vin'] == v]['month'].min() for v in mh['vin'].unique()}   # month the SoH curve starts at ~100% (battery new)
startm = {v: T100[v].to_period('M') for v in mh['vin'].unique()}            # GROUP KEY = 100%-SoH start month (= age-axis origin)
WYR = {v: config.warranty_for('mahindra', vmod.get(v, ''))[0] for v in mh['vin'].unique()}

def _reanchor(d):                                  # shift age axis so x=0 is the 100%-SoH month, not registration
    sh = (T100[d['vin']] - pd.Timestamp(d['reg'])).days / 365.25
    d['a_obs'] = d['a_obs'] - sh; d['a_fc'] = d['a_fc'] - sh

def show_degraders(wyr, model_name):
    pool = [v for v in mh['vin'].unique() if WYR[v] == wyr and nmo[v] >= MIN_MONTHS and dec[v] >= MIN_DECLINE]
    if not pool:
        print(f'{wyr}-yr ({model_name}): no vehicles meet the criteria'); return
    bm = Counter(startm[v] for v in pool).most_common(1)[0][0]     # the 100%-SoH start month they cluster in
    vins = sorted([v for v in pool if startm[v] == bm], key=lambda v: dec[v], reverse=True)[:N_SHOW]
    sel = [mh_d(v) for v in vins]
    for d in sel:
        d['a_fc'], d['s_fc'] = fc_to_eol(d['vin'], d['reg']); _reanchor(d)
    print(f'{wyr}-yr ({model_name}): {len(sel)} vehicles, 100% SoH from {bm} ->',
          [(d['vin'][-6:], f"-{dec[d['vin']]:.0f}pp", deg_tag(d['vin'])) for d in sel])
    ref = pd.Timestamp(int(np.median([pd.Timestamp(T100[d['vin']]).value for d in sel])))
    ymin = min(min(d['s_obs'].min(), d['s_fc'].min()) for d in sel) - 2
    xmax = max(max(d['a_fc'][-1] for d in sel), wyr) + 0.3
    fig = go.Figure()
    for i, d in enumerate(sel):
        col = PAL[i]; tail = d['label'].split()[0]; risk = soh_at_age(d, wyr) < 78
        fig.add_scatter(x=[0, d['a_obs'][0]], y=[100, d['s_obs'][0]], mode='lines', line=dict(color=col, width=1, dash='dot'),
                        opacity=0.5, legendgroup=tail, showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=d['a_obs'], y=d['s_obs'], mode='lines+markers', line=dict(color=col, width=1.8),
                        marker=dict(size=6, color=col), name=f"{tail} · {soh_at_age(d, wyr):.0f}% @ {wyr}yr" + (' ✓' if soh_at_age(d, wyr) >= 80 else ''),
                        legendgroup=tail, hovertemplate=tail + ' actual %{y:.1f}%<extra></extra>')
        fig.add_scatter(x=d['a_fc'], y=d['s_fc'], mode='lines', line=dict(color=col, width=2.2, dash='dash'),
                        legendgroup=tail, showlegend=False, hovertemplate=tail + ' forecast %{y:.1f}%<extra></extra>')
        if d['s_fc'][-1] <= 80.5:
            fig.add_scatter(x=[d['a_fc'][-1]], y=[d['s_fc'][-1]], mode='markers+text', legendgroup=tail, showlegend=False,
                            marker=dict(symbol='circle-open', size=9, color=col, line=dict(width=2)),
                            text=[f"{d['a_fc'][-1]:.1f}yr"], textposition='bottom center', textfont=dict(size=9, color=col),
                            hovertemplate=tail + ' hits 80%% at %{x:.1f} yr<extra></extra>')
        fig.add_scatter(x=[wyr], y=[soh_at_age(d, wyr)], mode='markers', legendgroup=tail, showlegend=False,
                        marker=dict(symbol='triangle-down', size=11, color=('crimson' if risk else 'seagreen'), line=dict(color='black', width=0.6)),
                        hovertemplate=tail + f" @{wyr}yr warranty " + '%{y:.1f}%<extra></extra>')
    fig.add_hline(y=80, line=dict(color='red', width=1.3, dash='dot'), annotation_text='80% EoFL', annotation_position='bottom left')
    fig.add_vline(x=wyr, line=dict(color='dimgray', width=1.3, dash='dashdot'))
    fig.add_annotation(x=wyr, y=ymin + 1.5, text=f'{wyr}-yr warranty', showarrow=False, textangle=-90,
                       xanchor='right', yanchor='bottom', font=dict(color='dimgray', size=10))
    ages = list(range(0, int(np.ceil(xmax)) + 1))
    dates = [(ref + pd.Timedelta(days=a * 365.25)).strftime('%b %Y') for a in ages]
    fig.add_scatter(x=ages, y=[ymin] * len(ages), xaxis='x2', mode='markers', marker=dict(opacity=0), showlegend=False, hoverinfo='skip')
    fig.update_layout(height=560, width=1120, template='plotly_white', margin=dict(t=70),
                      title=dict(text=f"Mahindra {model_name} — {wyr}-yr warranty, well-observed degraders (100% SoH from {bm}); legend = vehicle · SoH retained @ warranty", y=0.97),
                      xaxis=dict(title='age since 100% SoH (years)', range=[0, xmax]),
                      xaxis2=dict(overlaying='x', side='top', range=[0, xmax], tickmode='array', tickvals=ages, ticktext=dates,
                                  title=dict(text='calendar (year-month)', standoff=6), tickfont=dict(size=9), title_font=dict(size=10)),
                      yaxis=dict(title='coulomb SoH (%)', range=[ymin, 101]),
                      legend=dict(title='vehicle · SoH @ warranty', x=1.01, y=1, font=dict(size=10)))
    fig.show()

show_degraders(3, 'Treo')        # 3-yr warranty
show_degraders(5, 'Zor Grand')   # 5-yr warranty

In [ ]:
# === Plot per 100%-SoH START month — and within each, 3-yr (Treo) & 5-yr (Zor Grand) SEPARATELY ===
# Grouped by the month the SoH curve starts at 100% (battery new), not registration. Never clubs warranty
# terms: one plot = one 100%-SoH start month + one warranty term. Age axis origin = the 100%-SoH month.
def show_month(period, wyr, model_name, exclude=()):
    vins = sorted([v for v in mh['vin'].unique()
                   if startm[v] == period and WYR[v] == wyr and nmo[v] >= MIN_MONTHS and dec[v] >= MIN_DECLINE],
                  key=lambda v: dec[v], reverse=True)[:N_SHOW]
    vins = [v for v in vins if v[-6:] not in exclude and v not in exclude]   # drop AFTER top-N pick: no backfill
    if len(vins) < 2:
        return
    sel = [mh_d(v) for v in vins]
    for d in sel:
        d['a_fc'], d['s_fc'] = fc_to_eol(d['vin'], d['reg']); _reanchor(d)
    print(f'100% SoH from {period} · {wyr}yr ({model_name}): {len(sel)} ->', [(d['vin'][-6:], f"-{dec[d['vin']]:.0f}pp", deg_tag(d['vin'])) for d in sel])
    ref = pd.Timestamp(int(np.median([pd.Timestamp(T100[d['vin']]).value for d in sel])))
    ymin = min(min(d['s_obs'].min(), d['s_fc'].min()) for d in sel) - 2
    xmax = max(max(d['a_fc'][-1] for d in sel), wyr) + 0.3
    fig = go.Figure()
    for i, d in enumerate(sel):
        col = PAL[i % len(PAL)]; tail = d['label'].split()[0]; risk = soh_at_age(d, wyr) < 78
        fig.add_scatter(x=[0, d['a_obs'][0]], y=[100, d['s_obs'][0]], mode='lines', line=dict(color=col, width=1, dash='dot'),
                        opacity=0.5, legendgroup=tail, showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=d['a_obs'], y=d['s_obs'], mode='lines+markers', line=dict(color=col, width=1.8),
                        marker=dict(size=6, color=col), name=f"{tail} · {soh_at_age(d, wyr):.0f}% @ {wyr}yr" + (' ✓' if soh_at_age(d, wyr) >= 80 else ''),
                        legendgroup=tail, hovertemplate=tail + ' actual %{y:.1f}%<extra></extra>')
        fig.add_scatter(x=d['a_fc'], y=d['s_fc'], mode='lines', line=dict(color=col, width=2.2, dash='dash'),
                        legendgroup=tail, showlegend=False, hovertemplate=tail + ' forecast %{y:.1f}%<extra></extra>')
        if d['s_fc'][-1] <= 80.5:
            fig.add_scatter(x=[d['a_fc'][-1]], y=[d['s_fc'][-1]], mode='markers+text', legendgroup=tail, showlegend=False,
                            marker=dict(symbol='circle-open', size=9, color=col, line=dict(width=2)),
                            text=[f"{d['a_fc'][-1]:.1f}yr"], textposition='bottom center', textfont=dict(size=9, color=col),
                            hovertemplate=tail + ' hits 80%% at %{x:.1f} yr<extra></extra>')
        fig.add_scatter(x=[wyr], y=[soh_at_age(d, wyr)], mode='markers', legendgroup=tail, showlegend=False,
                        marker=dict(symbol='triangle-down', size=11, color=('crimson' if risk else 'seagreen'), line=dict(color='black', width=0.6)),
                        hovertemplate=tail + f" @{wyr}yr warranty " + '%{y:.1f}%<extra></extra>')
    fig.add_hline(y=80, line=dict(color='red', width=1.3, dash='dot'), annotation_text='80% EoFL', annotation_position='bottom left')
    fig.add_vline(x=wyr, line=dict(color='dimgray', width=1.3, dash='dashdot'))
    fig.add_annotation(x=wyr, y=ymin + 1.5, text=f'{wyr}-yr warranty', showarrow=False, textangle=-90,
                       xanchor='right', yanchor='bottom', font=dict(color='dimgray', size=10))
    ages = list(range(0, int(np.ceil(xmax)) + 1))
    dates = [(ref + pd.Timedelta(days=a * 365.25)).strftime('%b %Y') for a in ages]
    fig.add_scatter(x=ages, y=[ymin] * len(ages), xaxis='x2', mode='markers', marker=dict(opacity=0), showlegend=False, hoverinfo='skip')
    fig.update_layout(height=540, width=1120, template='plotly_white', margin=dict(t=70),
                      title=dict(text=f"Mahindra {model_name} — {wyr}-yr warranty · 100% SoH from {period}: well-observed degraders (legend = vehicle · SoH retained @ warranty)", y=0.97),
                      xaxis=dict(title='age since 100% SoH (years)', range=[0, xmax]),
                      xaxis2=dict(overlaying='x', side='top', range=[0, xmax], tickmode='array', tickvals=ages, ticktext=dates,
                                  title=dict(text='calendar (year-month)', standoff=6), tickfont=dict(size=9), title_font=dict(size=10)),
                      yaxis=dict(title='coulomb SoH (%)', range=[ymin, 101]),
                      legend=dict(title='vehicle · SoH @ warranty', x=1.01, y=1, font=dict(size=10)))
    fig.show()

qual = [v for v in mh['vin'].unique() if nmo[v] >= MIN_MONTHS and dec[v] >= MIN_DECLINE]
months_2023 = sorted({startm[v] for v in qual if startm[v].year == 2023})
print('2023 100%-SoH start months with well-observed degraders:', [str(p) for p in months_2023])
for p in months_2023:
    for wyr, name in [(3, 'Treo'), (5, 'Zor Grand')]:   # 3-yr and 5-yr ALWAYS plotted separately
        show_month(p, wyr, name)

In [ ]:
# Treo · 3-yr warranty · 100% SoH from 2023-03 — same well-observed-degrader plot, with L22079 removed
show_month(pd.Period('2023-03', 'M'), 3, 'Treo', exclude={'L22079'})

## Same registration-date cohorts (registration-month grouping)

Complementary to the 100%-SoH-start grouping above — the **same** well-observed degraders, but grouped by
**registration month** (vehicles sharing a registration date). Age axis = **age since registration**
(registration-anchored), so the dotted lead-in shows the gap before telemetry begins. 3-yr (Treo) and
5-yr (Zor Grand) are still plotted **separately** (never clubbed); legend = `vehicle · SoH retained @ warranty`.

In [ ]:
# === SAME REGISTRATION-DATE cohorts: well-observed degraders grouped by REGISTRATION month ===
# Complementary to the 100%-SoH grouping above. Here vehicles share a registration month; age axis =
# age since registration (registration-anchored, NO re-anchor — telemetry may begin a few months after
# registration, so markers start a little after age 0). 3-yr (Treo) & 5-yr (Zor Grand) ALWAYS separate.
regm = {v: REGD[v].to_period('M') for v in mh['vin'].unique()}     # REGISTRATION month = the calendar cohort

def show_month_reg(period, wyr, model_name, exclude=()):
    vins = sorted([v for v in mh['vin'].unique()
                   if regm[v] == period and WYR[v] == wyr and nmo[v] >= MIN_MONTHS and dec[v] >= MIN_DECLINE],
                  key=lambda v: dec[v], reverse=True)[:N_SHOW]
    vins = [v for v in vins if v[-6:] not in exclude and v not in exclude]   # drop AFTER top-N pick: no backfill
    if len(vins) < 2:
        return
    sel = [mh_d(v) for v in vins]
    for d in sel:
        d['a_fc'], d['s_fc'] = fc_to_eol(d['vin'], d['reg'])        # registration-anchored forecast (no _reanchor)
    print(f'registered {period} · {wyr}yr ({model_name}): {len(sel)} ->', [(d['vin'][-6:], f"-{dec[d['vin']]:.0f}pp", deg_tag(d['vin'])) for d in sel])
    ref = pd.Timestamp(int(np.median([pd.Timestamp(d['reg']).value for d in sel])))
    ymin = min(min(d['s_obs'].min(), d['s_fc'].min()) for d in sel) - 2
    xmax = max(max(d['a_fc'][-1] for d in sel), wyr) + 0.3
    fig = go.Figure()
    for i, d in enumerate(sel):
        col = PAL[i % len(PAL)]; tail = d['label'].split()[0]; risk = soh_at_age(d, wyr) < 78
        fig.add_scatter(x=[0, d['a_obs'][0]], y=[100, d['s_obs'][0]], mode='lines', line=dict(color=col, width=1, dash='dot'),
                        opacity=0.5, legendgroup=tail, showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=d['a_obs'], y=d['s_obs'], mode='lines+markers', line=dict(color=col, width=1.8),
                        marker=dict(size=6, color=col), name=f"{tail} · {soh_at_age(d, wyr):.0f}% @ {wyr}yr" + (' ✓' if soh_at_age(d, wyr) >= 80 else ''),
                        legendgroup=tail, hovertemplate=tail + ' actual %{y:.1f}%<extra></extra>')
        fig.add_scatter(x=d['a_fc'], y=d['s_fc'], mode='lines', line=dict(color=col, width=2.2, dash='dash'),
                        legendgroup=tail, showlegend=False, hovertemplate=tail + ' forecast %{y:.1f}%<extra></extra>')
        if d['s_fc'][-1] <= 80.5:
            fig.add_scatter(x=[d['a_fc'][-1]], y=[d['s_fc'][-1]], mode='markers+text', legendgroup=tail, showlegend=False,
                            marker=dict(symbol='circle-open', size=9, color=col, line=dict(width=2)),
                            text=[f"{d['a_fc'][-1]:.1f}yr"], textposition='bottom center', textfont=dict(size=9, color=col),
                            hovertemplate=tail + ' hits 80%% at %{x:.1f} yr<extra></extra>')
        fig.add_scatter(x=[wyr], y=[soh_at_age(d, wyr)], mode='markers', legendgroup=tail, showlegend=False,
                        marker=dict(symbol='triangle-down', size=11, color=('crimson' if risk else 'seagreen'), line=dict(color='black', width=0.6)),
                        hovertemplate=tail + f" @{wyr}yr warranty " + '%{y:.1f}%<extra></extra>')
    fig.add_hline(y=80, line=dict(color='red', width=1.3, dash='dot'), annotation_text='80% EoFL', annotation_position='bottom left')
    fig.add_vline(x=wyr, line=dict(color='dimgray', width=1.3, dash='dashdot'))
    fig.add_annotation(x=wyr, y=ymin + 1.5, text=f'{wyr}-yr warranty', showarrow=False, textangle=-90,
                       xanchor='right', yanchor='bottom', font=dict(color='dimgray', size=10))
    ages = list(range(0, int(np.ceil(xmax)) + 1))
    dates = [(ref + pd.Timedelta(days=a * 365.25)).strftime('%b %Y') for a in ages]
    fig.add_scatter(x=ages, y=[ymin] * len(ages), xaxis='x2', mode='markers', marker=dict(opacity=0), showlegend=False, hoverinfo='skip')
    fig.update_layout(height=540, width=1120, template='plotly_white', margin=dict(t=70),
                      title=dict(text=f"Mahindra {model_name} — {wyr}-yr warranty · REGISTERED {period}: well-observed degraders (legend = vehicle · SoH retained @ warranty)", y=0.97),
                      xaxis=dict(title='age since registration (years)', range=[0, xmax]),
                      xaxis2=dict(overlaying='x', side='top', range=[0, xmax], tickmode='array', tickvals=ages, ticktext=dates,
                                  title=dict(text='calendar (year-month)', standoff=6), tickfont=dict(size=9), title_font=dict(size=10)),
                      yaxis=dict(title='coulomb SoH (%)', range=[ymin, 101]),
                      legend=dict(title='vehicle · SoH @ warranty', x=1.01, y=1, font=dict(size=10)))
    fig.show()

reg_months = sorted({regm[v] for v in qual})
print('REGISTRATION months with well-observed degraders:', [str(p) for p in reg_months])
for p in reg_months:
    for wyr, name in [(3, 'Treo'), (5, 'Zor Grand')]:   # 3-yr and 5-yr ALWAYS plotted separately
        show_month_reg(p, wyr, name)